# Phase 6: Congestion Agent (Subscriber + Publisher)

This notebook subscribes to queue-state updates and publishes congestion-state updates when congestion starts or ends.

In [1]:
# Cell 2 purpose: Import modules and ensure src is available from notebooks/.
import json
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path('../src').resolve()))

from simulated_city import mqtt
from simulated_city.config import load_config
from simulated_city.congestion import CongestionPolicy, build_congestion_from_queue_state
from simulated_city.topic_schema import topic_congestion_state, topic_queue_state

In [5]:
# Cell 3 purpose: Load config, create policy, and connect MQTT client.
app_config = load_config()
if app_config.halftime is None:
    raise ValueError('Missing halftime section in config.yaml')

policy = CongestionPolicy(
    queue_people_per_line_threshold=app_config.halftime.blocking.queue_people_per_line_threshold,
    lines_considered=app_config.halftime.blocking.lines_considered,
)
queue_topic = topic_queue_state()
congestion_topic = topic_congestion_state()
mqtt_client = mqtt.connect_mqtt(app_config.mqtt, client_id_suffix='congestion-agent')

print(f'Connected to MQTT broker at {app_config.mqtt.host}:{app_config.mqtt.port}')
print(f'Subscribed topic: {queue_topic}')
print(f'Publish topic: {congestion_topic}')

Connected to MQTT broker at 127.0.0.1:1883
Subscribed topic: stadium/a4/halftime/state/queues
Publish topic: stadium/a4/halftime/state/congestion


In [6]:
# Cell 4 purpose: Subscribe to queue-state updates and publish congestion changes.
received_queue_states = []
published_congestion_states = []
last_is_congested = None

def _on_message(client, userdata, msg):
    global last_is_congested
    try:
        payload = json.loads(msg.payload.decode('utf-8'))
    except json.JSONDecodeError:
        return

    received_queue_states.append(payload)
    congestion_payload = build_congestion_from_queue_state(queue_state_payload=payload, policy=policy)
    if congestion_payload is None:
        return

    is_congested = bool(congestion_payload.get('zone_a_blocked', False) or congestion_payload.get('zone_b_blocked', False))
    if last_is_congested is None or is_congested != last_is_congested:
        ok = mqtt.publish_json_checked(client, congestion_topic, congestion_payload, qos=1)
        if ok:
            published_congestion_states.append(congestion_payload)
            last_is_congested = is_congested

mqtt_client.on_message = _on_message
mqtt_client.subscribe(queue_topic, qos=1)
print('Subscription started. Waiting for incoming queue-state events...')

Subscription started. Waiting for incoming queue-state events...


In [4]:
# Cell 5 purpose: Run short loop, print summary, and disconnect.
run_for_s = 30
start = time.time()
while time.time() - start < run_for_s:
    time.sleep(0.5)

print(f'Received queue-state events: {len(received_queue_states)}')
print(f'Published congestion-state changes: {len(published_congestion_states)}')
if published_congestion_states:
    print('Last published congestion payload:')
    print(published_congestion_states[-1])

connector = getattr(mqtt_client, '_simcity_connector', None)
if connector is not None:
    connector.disconnect()
    print('Disconnected from MQTT broker.')

Received queue-state events: 181
Published congestion-state changes: 1
Last published congestion payload:
{'schema_version': '1.0', 'run_id': 'a4-run-eb25aa71', 'timestamp_s': 0, 'zone_a_blocked': False, 'zone_b_blocked': False}


Disconnected from MQTT broker (reason=Normal disconnection). Reconnecting...


Disconnected from MQTT broker.
